# Upload tiles to OMERO

This notebook uploads a set of brightfield tiles to omero and parses the necessary metadata.

In [2]:
import omero
import os
from pathlib import Path
from skimage import io
import numpy as np
from bioio import BioImage
import ezomero
from omero.model.enums import UnitsLength

import tqdm

from functions import load_pos_file, get_tile_metadata

Log into omero:

In [4]:
host = 'omero-int.biotec.tu-dresden.de'
port = 4064
user = 'johamuel'
group = 'Grapin-Botton Lab'
secure=True

conn = ezomero.connect(host=host, user=user, secure=secure, port=port, group=group)

In [27]:
root = r'Z:\Collaborations\aif-botton-collaboration\Johannes'
tile_folder = os.path.join(root, "tile_images", "20250206_Well1_BF_multiPos_cropped_1536pxCropped_25percentOverlap_2D_1")
tile_folder

'Z:\\Collaborations\\aif-botton-collaboration\\Johannes\\tile_images\\20250206_Well1_BF_multiPos_cropped_1536pxCropped_25percentOverlap_2D_1'

In [41]:
os.path.join(root, "tile_images", "Well1_pos-list_microwellsFull.pos")

'Z:\\Collaborations\\aif-botton-collaboration\\Johannes\\tile_images\\Well1_pos-list_microwellsFull.pos'

In [55]:
# Define the path to your JSON file
file_path = os.path.join(root, "tile_images", "Well1_pos-list_microwellsFull.pos")

In [28]:
image_filenames = [os.path.join(tile_folder, f) for f in os.listdir(tile_folder) if os.path.isdir(os.path.join(tile_folder, f))]

In [29]:
dataset_id = 378
channel_names = {1: 'Brightfield'}

In [ ]:
for single_tile_folder in tqdm.tqdm(image_filenames):
    _folder = os.path.join(tile_folder, single_tile_folder)
    slice_filenames = [os.path.join(_folder, f) for f in os.listdir(_folder) if f.endswith('.tif')]
    tile_stack = np.stack([io.imread(f) for f in slice_filenames])[None, None, :]  # add singleton dimensions for channel and time

    # post image and channel names
    image_id = ezomero.post_image(conn, image=tile_stack, dataset_id=dataset_id, image_name=Path(single_tile_folder).stem, dim_order='tczyx', across_groups=False)
    conn.setChannelNames(data_type='Image', ids=[image_id], nameDict=channel_names)

    # set pixel sizes
    pxs = BioImage(slice_filenames[0]).physical_pixel_sizes
    unit_x = omero.model.LengthI(pxs.X, UnitsLength.MICROMETER)
    unit_y = omero.model.LengthI(pxs.Y, UnitsLength.MICROMETER)
    unit_z = omero.model.LengthI(10, UnitsLength.MICROMETER)  #z-scaping = 10microns

    image_object = conn.getObject("Image", image_id)
    primary_pixels_object = image_object.getPrimaryPixels()._obj
    primary_pixels_object.setPhysicalSizeX(unit_x)
    primary_pixels_object.setPhysicalSizeY(unit_y)
    primary_pixels_object.setPhysicalSizeZ(unit_z)
    conn.getUpdateService().saveObject(primary_pixels_object)

    # write metadata
    fname = Path(single_tile_folder).stem
    col = int(fname.split('_')[-2])
    row = int(fname.split('_')[-1])
    tile_metadata = get_tile_metadata(json_data, grid_col=col, grid_row=row)

    metadata_omero = {
        'GRID_COL': tile_metadata['GRID_COL'],
        'GRID_ROW': tile_metadata['GRID_ROW'],
        'XYStage_X': tile_metadata['DEVICES'][0]['X'],
        'XYStage_Y': tile_metadata['DEVICES'][0]['Y'],
        'ZStage_X': tile_metadata['DEVICES'][1]['X'],
    }

    map_annotation_id = ezomero.post_map_annotation(
        conn,
        object_type='Image',
        object_id=image_id,
        kv_dict=metadata_omero,
        ns='openmicroscopy.org/omero/client/mapAnnotation')

  0%|          | 1/324 [00:07<37:59,  7.06s/it]

object group 109


  1%|          | 2/324 [00:13<36:23,  6.78s/it]

object group 109


  1%|          | 3/324 [00:21<39:01,  7.30s/it]

...

object group 109


 27%|██▋       | 86/324 [11:19<25:10,  6.35s/it]

object group 109


 27%|██▋       | 86/324 [11:27<31:43,  8.00s/it]


KeyboardInterrupt: 